## Slurm Files, and Using the HPC Cluster

All of the steps I've taken so far, from downloading raw data to the PCA, have been performed on one sample from the knockout group (SRR36691286). But to recreate the actual analysis I need to use data from all four samples, and compare gene expression in different treatment groups. 

To download and analyze this massive amount of data, I requested an account on my lab's HPC cluster. Now that I got it set up I can rerun the downloading, quality control, and alignment for all four samples at once! 

Jobs for the HPC are submitted as Slurm files, which I'll save in a new folder alongside my original bash scripts.

### SBATCH Directives

At the top of each of these Slurm files is instructions called SBATCH directives. A set of instructions like this will be at the top of each file:

In [ ]:
#!/bin/tcsh
#SBATCH --job-name=ptp1b_job_    # Name for the job
#SBATCH --nodes=1                # Number of HPC nodes to use
#SBATCH --ntasks=1               # Number of tasks
#SBATCH --cpus-per-task=1        # Number of CPUs to use per task
#SBATCH --mem=1gb                # Amount of memory to use
#SBATCH --time=10:00:00          # Amount of time to give for the job
#SBATCH -e ptp1b_job.err         # Name for error file
#SBATCH -o ptp1b_job.out         # Name for output file

For each of my scripts the directives will look something like this, using my SRAToolkit script as an example:

In [ ]:
#!/bin/bash
#SBATCH --job-name=ptp1b_download
#SBATCH --nodes=1                                 # Nothing I'm doing requires more than 1 node
#SBATCH --ntasks=1                                # One task/process
#SBATCH --cpus-per-task=4                         # 4 CPUs is a reasonably small amount to request
#SBATCH --mem=16gb                                # 16gb memory is also reasonable
#SBATCH --time=10:00:00                           # This definitely won't take 10 hours, but just in case
#SBATCH -e ~/ptp1b_ad/logs/ptp1b_download.err     # Custom name and filepath for any error or output files
#SBATCH -o ~/ptp1b_ad/logs/ptp1b_download.out

After this I will remake each of my scripts, but expanding them to incorporate all four samples at once! This starts with downloading all four SRA accessions.

### Downloading

There are a few changes to using SRA Toolkit to download fastq files, relative to my first script.

To start, I have to load modules on the HPC. EB5 is a package with a lot of common scientific tools on it, and I'll also load SRA Toolkit by name to be safe:

In [ ]:
module load EB5
module load SRA-Toolkit

The instructions on my lab's HPC website recommend prefetching the samples, downloading them in the compressed .sra format before using fasterq-dump:

In [ ]:
prefetch -O ~/ptp1b_ad/data/sra SRR36691286 SRR36691287 SRR36691288 SRR36691289
# -O: Output location (new folder for sra files)

I also need to make sure this runs in the directory for this project, and that the directories for data and fastq files exist.

In [ ]:
# Start in the main project directory
cd ~/ptp1b_ad
# "~/": Absolute path starting from home directory

# Make data and fastq file directories
mkdir -p data/fastq
# -p: Create parent directory ('data') alongside new directory ('fastq') if necessary

Last is a set of instructions running the SRA Toolkit on each of the four files. Example for one file:

In [ ]:
# Run fasterq
fasterq-dump -e 4 -p -O data/fastq data/sra/SRR36691286/SRR36691286.sra

# Rename the output files based on treatment group
mv data/fastq/SRR36691286_1.fastq data/fastq/KO_SRR36691286_1.fastq
mv data/fastq/SRR36691286_2.fastq data/fastq/KO_SRR36691286_2.fastq

# Zip the files
gzip data/fastq/KO_SRR36691286_1.fastq
gzip data/fastq/KO_SRR36691286_2.fastq

### FastQC

Preparing a Slurm script for FastQC is pretty similar. It involves the module for FastQC, the same starting directory, and a new directory for output html files:

In [ ]:
module load EB5
module load FastQC

cd ~/ptp1b_ad
mkdir -p data/fastqc

After that it is easy to run on each sample:

In [ ]:
fastqc data/fastq/KO_SRR36691286_1.fastq.gz data/fastq/KO_SRR36691286_2.fastq.gz -o data/fastqc
fastqc data/fastq/KO_SRR36691287_1.fastq.gz data/fastq/KO_SRR36691287_2.fastq.gz -o data/fastqc
fastqc data/fastq/AD_SRR36691288_1.fastq.gz data/fastq/AD_SRR36691288_2.fastq.gz -o data/fastqc
fastqc data/fastq/AD_SRR36691289_1.fastq.gz data/fastq/AD_SRR36691289_2.fastq.gz -o data/fastqc

### Quantification

This is the trickiest part, but luckily I have already done a lot of the work finding reference files that can be easily transported over to the HPC.